In [0]:
from pyspark.sql import functions as F
# -----------------------------
# 1️⃣ Read CSV file from volume and load into table
# -----------------------------
csv_path = "/Volumes/workspace/demo/mmm_data/sample_data.csv 1.csv"

df_csv = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(csv_path)

display()
df = df_csv.select(
    [F.col(col).alias(col.upper()) for col in df_csv.columns]
)

display(df)

spark.sql("CREATE DATABASE IF NOT EXISTS demo")

df.write.format("delta").option("mergeSchema", "true").mode("overwrite").saveAsTable("demo.retail_media")

print("✅ Sample data loaded into `demo.retail_media` successfully!")

In [0]:
# Notebook cell 1 — install runtime libs (run on cluster)
%pip install langchain-core databricks-langchain langgraph-supervisor mlflow plotly



In [0]:
dbutils.library.restartPython()

In [0]:
from databricks_langchain import ChatDatabricks, DatabricksFunctionClient, set_uc_function_client
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor
from langgraph.graph.state import CompiledStateGraph
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)


In [0]:
# agentic_with_supervisor.py
import json
import re
from uuid import uuid4
from typing import Dict, Any, Generator
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# Databricks LangChain / LangGraph imports
from databricks_langchain import ChatDatabricks, DatabricksFunctionClient, set_uc_function_client
from langchain.agents import create_agent
from langgraph_supervisor import create_supervisor
from langgraph.graph.state import CompiledStateGraph
from mlflow.pyfunc import ResponsesAgent
from mlflow.types.responses import (
    ResponsesAgentRequest,
    ResponsesAgentResponse,
    ResponsesAgentStreamEvent,
    output_to_responses_items_stream,
    to_chat_completions_input,
)

# -----------------------------
# CONFIG
# -----------------------------
LLM_ENDPOINT_NAME = "databricks-meta-llama-3-3-70b-instruct"   # change to your desired model
TABLE_NAME = "demo.retail_media"
MAX_SQL_ROWS = 2000
SAFE_ROW_LIMIT = 1000
# -----------------------------

# initialize databricks function client and LLM
client = DatabricksFunctionClient()
set_uc_function_client(client)
llm = ChatDatabricks(endpoint=LLM_ENDPOINT_NAME)

# -----------------------------
# Simple prompt builder (workaround)
# -----------------------------
def make_prompt(template: str, question: str) -> str:
    return template.replace("{question}", question)

SQL_PROMPT_TEXT = """
You are an expert SQL generator for Databricks Delta tables.

Available table: `{table_name}` with columns:
DATE, ZONE, REGION, COUNTRY, RETAIL_CHANNEL, RETAILER_NAME, MANUFACTURER, PRODUCT_FAMILY, SPECIES, BRAND, SUB_BRAND, SKU_NAME, MARKETING_CHANNEL, CAMPAIGN_NAME, METRIC, VALUE

Rules:
- Return a single VALID Databricks SQL SELECT statement, no explanation, no markdown fences.
- Use uppercase for SQL keywords (SELECT, FROM, WHERE, GROUP BY, ORDER BY).
- Use ISO date format 'YYYY-MM-DD' when filtering by date.
- If ambiguous, produce a conservative aggregation (SUM(VALUE) with GROUP BY).
- Always include an ORDER BY.
- Do NOT include destructive statements.

User question:
{question}
""".strip().replace("{table_name}", TABLE_NAME)

CHART_PROMPT_TEXT = """
You are a visualization expert. You will receive:
- a short user request (the user's original NL question)
- a small JSON sample of the query results (first 5 rows)

Your job:
1. Choose the best visualization (LINE for date-series, BAR for categorical aggregates, PIE for simple share breakdowns, HEATMAP for 2D matrix).
2. Return ONLY executable Python code (no explanations) that:
   - expects a pandas DataFrame named `df` (full results)
   - builds a plotly figure assigned to variable `fig`
   - calls `fig.show()` at the end
3. Make the code robust (coerce date columns, handle missing columns).
"""

# -----------------------------
# LLM call helper (tries common invocations)
# -----------------------------
def call_llm_return_text(prompt: str) -> str:
    """
    Best-effort call wrapper for ChatDatabricks model; returns text content.
    """
    # try generate
    try:
        if hasattr(llm, "generate"):
            out = llm.generate([{"role":"user","content": prompt}])
            if hasattr(out, "generations"):
                gens = out.generations
                if isinstance(gens, (list, tuple)) and len(gens) > 0:
                    g0 = gens[0]
                    if isinstance(g0, (list, tuple)):
                        cand = getattr(g0[0], "text", str(g0[0]))
                    else:
                        cand = getattr(g0, "text", str(g0))
                    return str(cand)
            return str(out)
    except Exception:
        pass

    # try callable
    try:
        if callable(llm):
            out = llm(prompt)
            if isinstance(out, str):
                return out
            if hasattr(out, "content"):
                return out.content
            if hasattr(out, "text"):
                return out.text
            return str(out)
    except Exception:
        pass

    # other common methods
    for fn in ("invoke", "run", "predict", "chat", "generate_text", "complete"):
        if hasattr(llm, fn):
            try:
                out = getattr(llm, fn)(prompt)
                if isinstance(out, str):
                    return out
                if hasattr(out, "content"):
                    return out.content
                if hasattr(out, "text"):
                    return out.text
                return str(out)
            except Exception:
                continue

    raise RuntimeError("Unable to call ChatDatabricks LLM; inspect dir(llm).")

# -----------------------------
# Safe SQL runner
# -----------------------------
def run_sql(query: str, max_rows: int = MAX_SQL_ROWS) -> Dict[str, Any]:
    if not isinstance(query, str):
        raise ValueError("query must be a string")
    q_upper = query.upper()
    forbidden = ["DROP ", "DELETE ", "TRUNCATE ", "ALTER ", "SHUTDOWN", "GRANT ", "REVOKE ", "CREATE TABLE", "CREATE DATABASE"]
    for kw in forbidden:
        if kw in q_upper:
            raise ValueError(f"Refusing to run query containing forbidden keyword: {kw.strip()}")
    if "SELECT" not in q_upper:
        raise ValueError("Only SELECT queries are allowed.")
    if "LIMIT" not in q_upper:
        # query = f"SELECT * FROM ({query.rstrip(';')}) __q LIMIT {int(max_rows)}"
        query = f"SELECT * FROM ({query.rstrip(';')})"
    # df = spark.sql(query).limit(int(max_rows))
    df = spark.sql(query)
    pdf = df.toPandas()
    return {"query": query, "rows": json.loads(pdf.to_json(orient="records", date_format="iso")), "rowcount": len(pdf), "columns": list(pdf.columns)}

# -----------------------------
# Subagent implementations (LLM wrappers)
# -----------------------------
def sql_agent_call(user_question: str) -> str:
    prompt = make_prompt(SQL_PROMPT_TEXT, user_question)
    out = call_llm_return_text(prompt)
    # extract first SELECT substring
    up = out.upper()
    if "SELECT" in up:
        idx = up.find("SELECT")
        cand = out[idx:]
        cand = re.sub(r"```(?:sql)?", "", cand, flags=re.IGNORECASE).strip()
        cand = re.sub(r"```$", "", cand).strip()
        return cand
    return out.strip()

def chart_agent_call(user_question: str, sample_json: str) -> str:
    prompt = CHART_PROMPT_TEXT + "\n\nUser question:\n" + user_question + "\n\nSample data:\n" + sample_json
    out = call_llm_return_text(prompt)
    out = re.sub(r"^```(?:python)?\s*", "", out)
    out = re.sub(r"\s*```$", "", out)
    return out.strip()

# -----------------------------
# Build supervisor (LangGraph) and wrapper Response Agent
# -----------------------------
# Create two placeholder agents for the supervisor to be aware of (names & descriptions)
agent_descriptions = [
    {"name": "sql-generator-agent", "description": "Generates SQL SELECT statements from natural language queries."},
    {"name": "chart-generator-agent", "description": "Creates visualization code (Plotly) from dataset samples and user intent."}
]

supervisor_prompt = f"""
You are a supervisor responsible for coordinating subagents to answer the user's question.
Available subagents:
- sql-generator-agent: generates a Databricks SQL SELECT statement for table `{TABLE_NAME}`.
- chart-generator-agent: given a small JSON sample and the user's question, returns executable Plotly Python code (variable 'fig' and fig.show()).

Supervisor instructions:
1) Read the user's request.
2) Decide which subagent(s) to call and in what order.
3) Ask the sql-generator-agent to produce the SQL when needed.
4) After SQL is executed by the environment, ask the chart-generator-agent to create visualization code using a sample of the results.
5) Prefer safe, read-only operations.
6) Return a short orchestration trace (which agent was used) and a final result pointer.
"""

# Create simple create_agent placeholders (we don't attach a prompt here; supervisor uses names/descriptions)
agent_objects = [
    create_agent(llm, tools=[], name="sql-generator-agent"),
    create_agent(llm, tools=[], name="chart-generator-agent")
]

compiled_supervisor: CompiledStateGraph = create_supervisor(
    agents=agent_objects,
    model=llm,
    prompt=supervisor_prompt,
    add_handoff_messages=False,
    output_mode="full_history"
).compile()

# ResponsesAgent wrapper (so you can call .predict as earlier)
class LangGraphResponsesAgent(ResponsesAgent):
    def __init__(self, compiled: CompiledStateGraph):
        self.compiled = compiled
    def predict(self, request: ResponsesAgentRequest) -> ResponsesAgentResponse:
        outputs = [ event.item for event in self.predict_stream(request) if event.type == "response.output_item.done" ]
        return ResponsesAgentResponse(output=outputs, custom_outputs=request.custom_inputs)
    def predict_stream(self, request: ResponsesAgentRequest) -> Generator[ResponsesAgentStreamEvent, None, None]:
        cc_msgs = to_chat_completions_input([i.model_dump() for i in request.input])
        first = True
        seen = set()
        for _, events in self.compiled.stream({"messages": cc_msgs}, stream_mode=["updates"]):
            new_msgs = [ msg for v in events.values() for msg in v.get("messages", []) if msg.id not in seen ]
            if first:
                seen.update(msg.id for msg in new_msgs[: len(cc_msgs)])
                new_msgs = new_msgs[len(cc_msgs):]
                first = False
            else:
                seen.update(msg.id for msg in new_msgs)
                node_name = tuple(events.keys())[0]
                yield ResponsesAgentStreamEvent(type="response.output_item.done", item=self.create_text_output_item(text=f"<name>{node_name}</name>", id=str(uuid4())))
            if len(new_msgs) > 0:
                yield from output_to_responses_items_stream(new_msgs)

# instantiate supervisor agent (main orchestrator)
SUPERVISOR_AGENT = LangGraphResponsesAgent(compiled_supervisor)

# -----------------------------
# Agentic pipeline using supervisor
# -----------------------------
def nl_to_chart_agentic(user_question: str):
    """
    1) Ask supervisor to decide actions.
    2) Use supervisor + subagents to perform: SQL generation -> execute -> chart generation -> render.
    Returns dict with sql, df, fig (if any), and trace info.
    """
    trace = {"supervisor_prompt": supervisor_prompt, "actions": []}

    # Step 1: Ask supervisor which subagent(s) to call first (we use a short query to the supervisor to get decision)
    # Build a small supervisory query: ask to choose the next step only
    supervisor_query = f"User question: {user_question}\n\nPlease respond with which subagent to call first: one of [sql-generator-agent, chart-generator-agent]. Only return the agent name."
    try:
        # Try invoking the compiled supervisor via the LLM directly to get a routing decision (best-effort)
        sup_resp = call_llm_return_text(supervisor_query)
        chosen = sup_resp.strip().splitlines()[0].strip()
        trace["actions"].append({"supervisor_decision": chosen, "raw": sup_resp})
    except Exception as e:
        # fallback: default to sql-generator-agent
        chosen = "sql-generator-agent"
        trace["actions"].append({"supervisor_decision": chosen, "error": str(e)})

    # If supervisor picked chart-agent first, we still need SQL to get data — route to SQL agent anyway
    # Step 2: Call SQL generator agent
    sql_text = sql_agent_call(user_question)
    trace["actions"].append({"sql_generated_raw": sql_text})
    # Extract SQL SELECT
    if "SELECT" not in sql_text.upper():
        # attempt to find SELECT substring
        up = sql_text.upper()
        idx = up.find("SELECT")
        if idx != -1:
            sql_text = sql_text[idx:]
    generated_sql = sql_text.strip()
    trace["actions"].append({"generated_sql": generated_sql})

    print("=== Generated SQL ===")
    print(generated_sql)

    # Step 3: Execute SQL
    sql_result = run_sql(generated_sql, max_rows=MAX_SQL_ROWS)
    rows = sql_result.get("rows", [])
    df = pd.DataFrame(rows)
    trace["actions"].append({"executed_rowcount": sql_result.get("rowcount", 0)})

    if df.empty:
        print("Query returned 0 rows.")
        return {"trace": trace, "sql": generated_sql, "df": df, "fig": None}

    # Step 4: Ask supervisor which chart-agent action to take (example: ask whether line/bar/pie)
    # We send sample and question
    sample_json = json.dumps(df.head(5).to_dict(orient="records"), default=str, indent=2)
    chart_decision_prompt = f"User question: {user_question}\nSample data (first 5 rows):\n{sample_json}\n\nWhich chart type should be used? Choose one of: LINE, BAR, PIE, HEATMAP. Return only the word."
    try:
        chart_choice_raw = call_llm_return_text(chart_decision_prompt)
        chart_choice = chart_choice_raw.strip().splitlines()[0].strip().upper()
    except Exception:
        chart_choice = "LINE" if any(c.lower() == "date" for c in df.columns) else "BAR"
    trace["actions"].append({"chart_choice": chart_choice})

    # Step 5: Call chart agent to get code
    chart_code = chart_agent_call(user_question, sample_json)
    trace["actions"].append({"chart_code_snippet": chart_code[:300] if chart_code else None})

    # Step 6: Execute chart code with restricted env (if model returned code)
    fig = None
    if chart_code:
        local_env = {"pd": pd, "px": px, "go": go, "df": df.copy()}
        safe_builtins = {"__builtins__": {"len": len, "range": range, "min": min, "max": max, "sum": sum}}
        try:
            exec(chart_code, safe_builtins, local_env)
            fig = local_env.get("fig", None)
            if fig is not None:
                try:
                    fig.show()
                except Exception:
                    display(fig)
        except Exception as e:
            trace["actions"].append({"chart_exec_error": str(e)})
            fig = None

    # Step 7: fallback heuristics if no fig
    if fig is None:
        date_col = next((c for c in df.columns if c.lower() in ("date", "dt", "day", "timestamp")), None)
        numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
        categorical_cols = [c for c in df.columns if c not in numeric_cols and c != date_col]
        try:
            if date_col and numeric_cols:
                ncol = numeric_cols[0]
                df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
                fig = px.line(df.sort_values(date_col), x=date_col, y=ncol, title=user_question)
            elif categorical_cols and numeric_cols:
                c = categorical_cols[0]
                n = numeric_cols[0]
                agg = df.groupby(c)[n].sum().reset_index().sort_values(n, ascending=False).head(20)
                fig = px.bar(agg, x=c, y=n, title=user_question)
            elif numeric_cols:
                n = numeric_cols[0]
                fig = px.histogram(df, x=n, title=user_question)
            else:
                display(df.head(50))
        except Exception as e:
            trace["actions"].append({"fallback_error": str(e)})

        if fig is not None:
            try:
                fig.show()
            except Exception:
                display(fig)

    return {"trace": trace, "sql": generated_sql, "df": df, "fig": fig}

# -----------------------------
# Example run
# -----------------------------
if __name__ == "__main__":
    q = "Show total SPENDS and total HH_GRPS for DOG species. Plot the  values for both metrics in a time series chart month and year wise. month year will be in x axis and y axis will be the values"
    out = nl_to_chart_agentic(q)
    print("Trace:", json.dumps(out["trace"], indent=2))
    print("Rows:", out["df"].shape if isinstance(out["df"], pd.DataFrame) else None)


In [0]:
%sql
SELECT 
  EXTRACT(YEAR FROM DATE) AS YEAR,
  SUM(CASE WHEN METRIC = 'SPENDS' THEN VALUE ELSE 0 END) AS TOTAL_SPENDS,
  SUM(CASE WHEN METRIC = 'HH_GRPS' THEN VALUE ELSE 0 END) AS TOTAL_HH_GRPS
FROM 
  demo.retail_media
WHERE 
  SPECIES = 'DOG'
GROUP BY 
  EXTRACT(YEAR FROM DATE)
ORDER BY 
  YEAR

In [0]:
%sql
SELECT 
  EXTRACT(YEAR FROM DATE) AS YEAR,
  SUM(CASE WHEN METRIC = 'spends' THEN VALUE ELSE 0 END) AS total_spends,
  SUM(CASE WHEN METRIC = 'hh_grps' THEN VALUE ELSE 0 END) AS total_hh_grps
FROM 
  demo.retail_media
GROUP BY 
  EXTRACT(YEAR FROM DATE)
ORDER BY 
  YEAR

In [0]:
Trace: {
  "supervisor_prompt": "\nYou are a supervisor responsible for coordinating subagents to answer the user's question.\nAvailable subagents:\n- sql-generator-agent: generates a Databricks SQL SELECT statement for table `demo.retail_media`.\n- chart-generator-agent: given a small JSON sample and the user's question, returns executable Plotly Python code (variable 'fig' and fig.show()).\n\nSupervisor instructions:\n1) Read the user's request.\n2) Decide which subagent(s) to call and in what order.\n3) Ask the sql-generator-agent to produce the SQL when needed.\n4) After SQL is executed by the environment, ask the chart-generator-agent to create visualization code using a sample of the results.\n5) Prefer safe, read-only operations.\n6) Return a short orchestration trace (which agent was used) and a final result pointer.\n",
  "actions": [
    {
      "supervisor_decision": "sql-generator-agent",
      "raw": "sql-generator-agent"
    },
    {
      "sql_generated_raw": "SELECT \n  EXTRACT(YEAR FROM DATE) AS YEAR,\n  EXTRACT(MONTH FROM DATE) AS MONTH,\n  SUM(CASE WHEN METRIC = 'SPENDS' THEN VALUE ELSE 0 END) AS TOTAL_SPENDS,\n  SUM(CASE WHEN METRIC = 'HH_GRPS' THEN VALUE ELSE 0 END) AS TOTAL_HH_GRPS\nFROM \n  demo.retail_media\nWHERE \n  SPECIES = 'DOG'\nGROUP BY \n  EXTRACT(YEAR FROM DATE),\n  EXTRACT(MONTH FROM DATE)\nORDER BY \n  YEAR,\n  MONTH"
    },
    {
      "generated_sql": "SELECT \n  EXTRACT(YEAR FROM DATE) AS YEAR,\n  EXTRACT(MONTH FROM DATE) AS MONTH,\n  SUM(CASE WHEN METRIC = 'SPENDS' THEN VALUE ELSE 0 END) AS TOTAL_SPENDS,\n  SUM(CASE WHEN METRIC = 'HH_GRPS' THEN VALUE ELSE 0 END) AS TOTAL_HH_GRPS\nFROM \n  demo.retail_media\nWHERE \n  SPECIES = 'DOG'\nGROUP BY \n  EXTRACT(YEAR FROM DATE),\n  EXTRACT(MONTH FROM DATE)\nORDER BY \n  YEAR,\n  MONTH"
    },
    {
      "executed_rowcount": 5
    },
    {
      "chart_choice": "LINE"
    },
    {
      "chart_code_snippet": "import pandas as pd\nimport plotly.express as px\n\n# Ensure date columns are datetime\ndf['DATE'] = pd.to_datetime(df[['YEAR', 'MONTH']].assign(DAY=1))\n\n# Create a line chart\nfig = px.line(df, x='DATE', y=['TOTAL_SPENDS', 'TOTAL_HH_GRPS'])\n\n# Update layout\nfig.update_layout(\n    title='Time Series Char"
    },
    {
      "chart_exec_error": "__import__ not found"
    }
  ]
}
Rows: (5, 4)


In [0]:
%sql
SELECT 
  EXTRACT(YEAR FROM DATE) AS YEAR,
  EXTRACT(MONTH FROM DATE) AS MONTH,
  SUM(CASE WHEN METRIC = 'SPENDS' THEN VALUE ELSE 0 END) AS TOTAL_SPENDS,
  SUM(CASE WHEN METRIC = 'HH_GRPS' THEN VALUE ELSE 0 END) AS TOTAL_HH_GRPS
FROM 
  demo.retail_media
WHERE 
  SPECIES = 'DOG'
GROUP BY 
  EXTRACT(YEAR FROM DATE),
  EXTRACT(MONTH FROM DATE)
ORDER BY 
  YEAR,
  MONTH